# Sylber-for-Khmer — Colab T4 smoke test

A minimal, fast run of [Sylber-Speech-Tokenizer](https://github.com/Pich09/Sylber-Speech-Tokenizer)'s
pipeline to check the idea works end-to-end, not to produce a final
result: Step 1 (data prep, ~40 min of audio) -> Step 2 (zero-shot
segmentation sanity check) -> the CTC probe (`src/train_ctc.py`), the
cheap ASR-viability test documented in `docs/post-benchmark-roadmap.md`.

**Before running**: `Runtime` -> `Change runtime type` -> select **T4 GPU**.
Then `Runtime` -> `Run all`.

**Deliberately small**: `MAX_SAMPLES = 300` (~40 min of audio, ~240/30/30
train/val/test utterances) and 8 CTC epochs — enough to confirm the
pipeline runs and get a directional CER number, not a statistically solid
result. Raise `MAX_SAMPLES` in section 2's data-prep cell once you've
confirmed everything runs clean. Full-scale runs (discrete-SLM comparison,
backbone fine-tuning) belong on the RTX 4070 machine per
`docs/run-roadmap.md` — this notebook doesn't attempt either.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

Colab already ships a CUDA-enabled `torch`, so `requirements.txt` installs
as-is rather than pulling a specific CUDA-index build (that's only needed
on a machine without a CUDA-matched wheel already present).

In [ ]:
import os

if os.path.isdir(".git"):
    print("Already inside the repo — pulling latest.")
    !git pull
elif os.path.isdir("Sylber-Speech-Tokenizer"):
    %cd Sylber-Speech-Tokenizer
    !git pull
else:
    !git clone https://github.com/Pich09/Sylber-Speech-Tokenizer.git
    %cd Sylber-Speech-Tokenizer

In [ ]:
!pip install -q -r requirements.txt

**If the next cell errors** (import error, or `cuda available: False`),
use `Runtime` -> `Restart session`, then run all cells again from the top
*without* re-running the `pip install` cell's `-q` output check — Colab
occasionally needs a restart to pick up packages installed after the
session's own preloaded ones.

In [ ]:
import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU visible — check Runtime > Change runtime type > T4 GPU, or Restart session and rerun."

### Patch: `sylber`/`torchtyping` incompatibility

`docs/path-a-encoder-comparison.md`'s "Pipeline notes" section records that
the unmaintained `torchtyping` dependency can crash on newer torch
(`Cannot subclass _TensorBase directly`) when `sylber` imports its unused
`SegmentSynthesis` class. This locates the installed package **without**
importing it first (so it works even if the import itself is what's
crashing) and patches the offending import defensively. No-op if your
torch/torchtyping combination doesn't hit the bug.

In [ ]:
import importlib.util
import re
import site
from pathlib import Path


def find_sylber_init():
    spec = importlib.util.find_spec("sylber")
    if spec and spec.origin:
        return Path(spec.origin)
    for sp in site.getsitepackages():
        cand = Path(sp) / "sylber" / "__init__.py"
        if cand.exists():
            return cand
    return None


init_path = find_sylber_init()
if init_path is None:
    print("Could not locate the installed sylber package — skipping patch (pip install may have failed; check the cell above).")
else:
    src = init_path.read_text()
    if "# patched-by-colab-notebook" in src:
        print("Already patched — nothing to do.")
    elif "SegmentSynthesis" not in src:
        print("No SegmentSynthesis import found in", init_path, "— nothing to patch.")
    else:
        patched = re.sub(
            r"^(.*\bSegmentSynthesis\b.*)$",
            r"try:\n    \1\nexcept Exception as _e:  # patched-by-colab-notebook: unused import, torchtyping incompatible with newer torch\n    print('sylber: skipped SegmentSynthesis import (', _e, ')')",
            src,
            flags=re.MULTILINE,
        )
        init_path.write_text(patched)
        print("Patched", init_path)

try:
    import sylber
    print("import sylber OK")
except Exception as e:
    print("import sylber still failing after patch attempt:", e)
    print("Open", init_path, "in the Colab file browser and comment out the SegmentSynthesis import by hand.")

## 2. Step 1 — data prep (small streamed subset)

`--max-samples` streams only the first N examples from the Hugging Face
dataset instead of downloading the full ~495GB/1,065h corpus, which
doesn't fit a Colab session. `MAX_SAMPLES = 300` is deliberately small —
utterances in this corpus average ~8.5s, so this is ~40 min of audio, just
enough to confirm the pipeline works and get a directional signal.

In [ ]:
MAX_SAMPLES = 300  # raise this (e.g. 3000-5000) for a firmer read once the pipeline is confirmed working
DATASET = "DDD-Cambodia/khmer-speech-dataset"
MANIFEST_NAME = "khmer_colab_smoketest_manifest.csv"

!python data/preprocessing/prepare_data.py \
    --dataset {DATASET} \
    --out data/khmer_colab_smoketest \
    --max-samples {MAX_SAMPLES} \
    --manifest-name {MANIFEST_NAME}

In [ ]:
import pandas as pd

MANIFEST_PATH = f"data/preprocessing/manifests/{MANIFEST_NAME}"
df = pd.read_csv(MANIFEST_PATH).fillna({"transcript": ""})
assert len(df) > 0, "Manifest is empty — data prep likely failed; check the cell above for errors."

empty_frac = (df["transcript"] == "").mean()
assert empty_frac < 0.5, (
    f"{empty_frac:.0%} of rows have an empty transcript — the dataset's transcript field likely "
    "doesn't match prepare_data.py's .get('transcript')/'transcription'/'text' fallback chain. "
    "Check the actual column names (e.g. https://datasets-server.huggingface.co/first-rows?dataset="
    f"{DATASET.replace('/', '%2F')}&config=default&split=train) and update that fallback chain, "
    "then rerun the data-prep cell above. Without this, the CTC probe below will silently skip "
    "every utterance (vocab_size=0, val_cer=0.0 looks like success but isn't)."
)

print(f"{len(df)} utterances, {df['duration_sec'].sum() / 3600:.2f}h total, {empty_frac:.0%} empty transcripts")
print(df["split"].value_counts())
df.head()

## 3. Sanity-check `sylber`'s output shape on one file

The one real assumption this code makes about the installed `sylber`
package (a `segment_features` key) — worth confirming before the next
steps rely on it.

In [ ]:
from sylber import Segmenter

_seg = Segmenter(model_ckpt="sylber")
_out = _seg(df.iloc[0]["path"], in_second=True)
print("Output keys:", list(_out.keys()) if isinstance(_out, dict) else type(_out))
assert isinstance(_out, dict) and "segment_features" in _out, (
    "Installed sylber's output has no 'segment_features' key — this code's assumption doesn't hold "
    "for this version; see README's 'Notes / open items' section for the marked spots to adjust."
)

import numpy as np
_feats = np.asarray(_out["segment_features"])
_transcript_len = len(df.iloc[0]["transcript"])
print(f"segment_features shape: {_feats.shape}  (dim 0 should be the syllable/token count, not a batch size of 1)")
print(f"first utterance: {_transcript_len} transcript chars, duration={df.iloc[0]['duration_sec']:.2f}s")
assert _feats.ndim == 2, (
    f"Expected segment_features to be 2-D (T, D) but got shape {_feats.shape} — if there's an extra "
    "leading batch dimension, train_ctc.py's feats.shape[0] check reads that batch size (likely 1) "
    "instead of the true token count T, which would make every utterance fail CTC's "
    "input_length >= target_length check regardless of actual segmentation quality."
)
del _seg, _out

## 4. Step 2 — zero-shot segmentation eval

Expected token rate is 4-5 Hz; the script flags PASS/WARNING automatically.

In [ ]:
!python src/segmentation.py eval \
    --manifest {MANIFEST_PATH} \
    --n-samples 30 \
    --out results/zero_shot_evaluation.txt

print(open("results/zero_shot_evaluation.txt").read())

## 5. CTC probe — the idea being tested

Trains a small supervised CTC decoder on top of each frozen encoder's
features and reports an error rate close to (but not exactly) character
error rate — labels are Unicode grapheme clusters, not raw codepoints,
since Khmer syllables are encoded as a base consonant plus combining marks
(coeng subscripts, vowel signs). Splitting on raw codepoints would inflate
label length ~2-4x relative to actual syllables, which fails CTC's
`input_length >= target_length` requirement against Sylber's ~4.76Hz token
rate for nearly every utterance — grapheme clusters keep label length
close to one per syllable instead.

This is the test validated at low data scales by Sylber 2.0's
"Low-Resource ASR" experiments (20-50h per language), unlike the
discrete-SLM comparison which needs hundreds/thousands of hours to be
meaningful. 8 epochs here is enough to confirm training runs and loss
moves, not to converge fully.

**If `n_used` stays at/near 0** in the training log below, check the
`skip_reasons` breakdown the results cell prints — `input_shorter_than_target`
dominating means try longer utterances or a bigger `MAX_SAMPLES`;
`empty_transcript`/`no_labelable_units` dominating means the dataset's
transcript field changed again (see section 2's manifest check).

In [ ]:
!python src/train_ctc.py train \
    --manifest {MANIFEST_PATH} \
    --encoder sylber \
    --epochs 8

In [ ]:
!python src/train_ctc.py train \
    --manifest {MANIFEST_PATH} \
    --encoder hubert \
    --checkpoint facebook/hubert-base-ls960 \
    --epochs 8

## 6. Results

In [ ]:
import json
from pathlib import Path

result_files = sorted(Path("results/downstream_eval").glob("ctc_probe_*.json"))
assert result_files, "No ctc_probe_*.json files found — the training cells above likely errored."

for p in result_files:
    r = json.loads(p.read_text())
    first_train = r["history"][0]["train"]
    last_val = r["history"][-1]["val"]
    print(f"{r['encoder']:8s} final_val_cer={r['final_val_cer']}  n_train={r['n_train']}  n_val={r['n_val']}")
    print(f"         last-epoch val used={last_val['n_used']} skipped={last_val['n_skipped']} skip_reasons={last_val['skip_reasons']}")
    if last_val["n_used"] == 0:
        print(f"         WARNING: every val utterance was skipped for {r['encoder']} — final_val_cer is not meaningful.")
    if first_train["n_used"] == 0 and first_train.get("length_skip_examples"):
        print(f"         epoch-1 train (encoder_output_shape, label_length) examples: {first_train['length_skip_examples']}")
        print("         a fixed/tiny leading shape dim (e.g. always 1) regardless of label_length points at a")
        print("         batch-dimension bug in the encoder adapter; a varying-but-still-too-small shape points")
        print("         at genuine zero-shot under-segmentation for this language instead.")

print("\nSmoke test complete — pipeline runs end-to-end.")
print("This is a directional signal only (small subset, few epochs). Raise MAX_SAMPLES")
print("in section 2's data-prep cell and epochs in section 5 for a firmer read, or move to")
print("the RTX 4070 machine per docs/post-benchmark-roadmap.md once you're ready to commit more time.")